# 1. Global UAP Sighting Patterns

## NUFORC Dataset: Initial Exploration and Preparation

This notebook focuses on the **National UFO Reporting Center (NUFORC)** dataset, which contains civilian-submitted reports of unusual or atmospheric observations. 

In recent government and scientific discussions, the term **UAP (Unidentified Anomalous Phenomena)** has replaced the older term **UFO**. The updated terminology reflects a broader scientific investigation into unexplained observations occurring in the **air, sea, or space**, and avoids the cultural stigma with the word "UFO". The focus of UAP research is on data collection, national security, and scientific analysis rather than assumptions about extraterrestrial origins. 

This project analyzes global sighting patterns using three primary datasets:
1. **Kaggle UFO Sightings Dataset (1906–2014)**  
   A large historical dataset of civilian-reported sightings.

2. **NUFORC Dataset (National UFO Reporting Center)**  
   A modern civilian reporting database containing detailed narrative descriptions of sightings.

3. **GEIPAN Dataset (Groupe d'Études et d'Informations sur les Phénomènes Aérospatiaux Non Identifiés)**  
   An official French government program responsible for investigating unidentified aerospace phenomena.

Each dataset has a different structure and level of detail. Because of these differences, each dataset is explored and cleaned in separate notebooks before being standardized and integrated into a unified relational database for analysis.

## 2. Purpose of This Notebook

The goal of this notebook is to perform an initial exploration and structural assessment of the NUFORC dataset before detailed cleaning and transformation. 

Specifically, this notebook will:
- Load the raw NUFORC dataset
- Inspect its structure, contents, and size of the dataset 
- Examine how sighting information is stored within the dataset.
- Identify key attributes embedded in the raw report text
- Define the fields that must be extracted and standardized

This notebook focuses on understanding the dataset structure and preparing a strategy for parsing the report data. Data extracted and transformation will occur in later steps of the project pipeline. 

## 3. Import Libraries

The following Python libraries are used for data loading, inspection, and early-stage preparation. 

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## 4. Load NUFORC Dataset

The NUFORC file is stored in the project's 'data/raw/' directory. 

At this stage, the objective is simply to confirm that the dataset loads correctly and to preview the raw structure of the data. 

The dataset currently contains two columns: 
- **sighting** — a unique identifier for each reported event
- **report** — a narrative text field containing multiple attributes embedded in a single string
 
Because key information such as location, time, object shape, and duration are embedded within the narrative report text, additional parsing will be required to convert this data into structured columns suitable for database storage and analysis. 

In [32]:
# Files:
nuforc = pd.read_csv("../data/raw/nuforc_flat.csv")

nuforc.head()

,sighting,report
0,114864,Occurred: 2014-09-21 13:00:00 Local\nLocation:...
1,126755,Occurred: 2015-12-18 13:00:00 Local\nLocation:...
2,106946,Occurred: 2014-02-05 21:00:00 Local\nLocation:...
3,161419,Occurred: 2019-05-16 12:00:00 Local\nLocation:...
4,18735,Occurred: 2001-07-29 23:59:00 Local\nLocation:...


## 5. Initial Dataset Inspection

Before cleaning the dataset, I will inspect its size, column names, and a small random sample of rows. This will help determine how much preprocessing will be required and whether the dataset structure aligns with the schema used in this project.

### Dataset Dimensions

In [33]:
nuforc.shape

(147890, 2)

**Interpretation:** The dataset contains 147,890 records and 2 columns. This confirms that the NUFORC dataset stores most case information within a single narrative report field rather than in a fully structured tabular format.

### Column Names

In [34]:
nuforc.columns

Index(['sighting', 'report'], dtype='object')

**Interpretation:** The dataset contains only two fields: `sighting`, which functions as a case identifier, and `report`, which stores multiple case attributes inside a narrative text field. This indicates that additional parsing will be required before the dataset can be integrated into the project database.

### 5 Sample Records

In [35]:
nuforc.sample(5, random_state=42)

,sighting,report
45564,143911,Occurred: 2018-11-26 20:02:00 Local\nLocation:...
111642,178414,Occurred: 2023-09-20 14:28:00 Local\nLocation:...
54239,92690,Occurred: 2012-09-10 15:15:00 Local\nLocation:...
145758,121404,Occurred: 2015-08-22 23:40:00 Local\nLocation:...
68331,89799,Occurred: 2012-06-21 00:00:00 Local\nLocation:...


**Interpretation**: The sample records show that observational details such as occurrence time, location, and descriptive narrative are embedded within the 'report' text field. This confirms that processing will require text parsing to extract structured fields for analysis.

## 6. Raw Dataset Structure

The NUFORC dataset differs significantly from the historical Kaggle dataset used earlier in this project.  
Instead of storing each attribute in its own column, the NUFORC dataset stores most sighting details inside a single narrative text field.

At this stage, the dataset contains two columns:

1. **sighting** — a unique identifier for each reported event  
2. **report** — a narrative text field containing multiple case attributes embedded within the narrative text.

Important information such as observation date, time, location, object shape, duration, summary details, and descriptive narrative is embedded within the `report` field rather than stored as separate structured variables.

Because of this structure, the next stage of processing will involve parsing the `report` field into individual structured columns that can be standardized and integrated into a relational database. 

## 7. Data Preparation Plan

To integrate the NUFORC dataset with the project's relational database schema, several attributes must be extracted from the narrative `report` field.

Because the dataset stores multiple case details within a single text field, a parsing process will be used to identify and extract structured attributes where possible.

The following structured fields will be derived from the report text when available:

- observation_date  
- year  
- city  
- region_state_province  
- country  
- shape  
- duration_seconds  
- summary  
- description  
- latitude (when available)  
- longitude (when available)  

Geographic coordinates (`latitude` and `longitude`) may appear within some NUFORC report entries. When coordinates are present in the narrative text, they will be extracted during the parsing stage.

However, coordinates are not consistently available for all reports. For records where latitude and longitude are missing, geographic coordinates will be generated during a later geocoding step using the standardized location fields.

## 8. Next Steps

The next phase of the workflow will involve creating a custom parsing function to extract structured attributes from the NUFORC 'report' text. 

This parsing process will:

- identify patterns within the narrative report
- extract standardized fields
- convert text-based information into structured columns
- prepare the dataset for cleaning, standardization, and later integration into the project's relational database. 

Once the extraction process is complete, the structured NUFORC dataset will move to the cleaning and standardization stage before later integration with the other project datasets for cross-dataset analysis. 

### Inspect Common Report Labels

Before building the parser function, the labeled fields within the NUFORC `report` text are inspected. This helps determine which attributes appear consistently enough to be extracted reliably and which fields may require fallback logic or later enrichment.

In [36]:
common_labels = [
    "Occurred:",
    "Location:",
    "Shape:",
    "No of observers:",
    "Characteristics:",
    "Summary:",
    "Text:"
]

for label in common_labels:
    count = nuforc["report"].str.contains(label, regex=False, na=False).sum()
    print(f"{label} {count}")

Occurred: 147890
Location: 147889
Shape: 141568
No of observers: 141231
Characteristics: 106262
Summary: 146999
Text: 146999


### Observations

The inspection confirms that several labeled fields appear consistently within the NUFORC narrative reports.

- `Occurred` appears in essentially all records and can be used to extract the observation date.
- `Location` also appears on nearly all records, making it a reliable source for location parsing.
- `Shape`, `Summary`, and `Text` are also highly consistent and are strong candidates for structured extraction.
- `No of observers` and `Characteristics` appear in many records, but not all, so they should be treated as optional fields.

These findings confirm that the NUFORC `report` field follows a semi-structured format. Because several labels appear consistently, the parser can use them as anchors for extracting structured fields from the narrative text. 

### Parsing Function

The following function parses the NUFORC narrative 'report' field and extracts structured attributes such as date, location, shape, and duration when available. 

In [37]:
# re = Regular Expressions ("regex") module
# Used for pattern matching and extracting structured information from text
import re

def parse_nuforc_report(report_text):
    """
    Extract structured fields from the NUFORC report narrative.
    Returns a dictionary with extracted attributes.
    """

# pd.isna --> Pandas DataFrame, pd.isna = used to detect missing values for an array-like object
    if pd.isna(report_text):
        return {}

    result = {}

    # Extract observation date
    date_match = re.search(r"Occurred:\s*([^\n]+)", report_text)
    if date_match:
        result["observation_date"] = date_match.group(1).strip()

    # Extract location
    location_match = re.search(r"Location:\s*([^\n]+)", report_text)
    if location_match:
        result["location_raw"] = location_match.group(1).strip()

    # Extract shape
    shape_match = re.search(r"Shape:\s*([^\n]+)", report_text)
    if shape_match:
        result["shape"] = shape_match.group(1).strip()

    # Extract duration
    duration_match = re.search(r"Duration:\s*([^\n]+)", report_text)
    if duration_match:
        result["duration_raw"] = duration_match.group(1).strip()

    return result

**Interpretation:** This initial parsing function focuses on the most consistently labeled fields identified in the previous inspection step. It extracts observation date, location, shape, and duration from the narrative `report` text and stores them as structured key-value pairs for later transformation into columns.

### Test Parser

In [38]:
# Testing the parser on several reports
for i in range(5):
    print(parse_nuforc_report(nuforc["report"].iloc[i]))

{'observation_date': '2014-09-21 13:00:00 Local', 'location_raw': 'Huntsville, TX, USA', 'shape': 'Rectangle', 'duration_raw': 'several seconds'}
{'observation_date': '2015-12-18 13:00:00 Local', 'location_raw': 'Sonoma, CA, USA', 'shape': 'Sphere', 'duration_raw': '2 minutes'}
{'observation_date': '2014-02-05 21:00:00 Local', 'location_raw': 'Hershey, PA, USA', 'shape': 'Light', 'duration_raw': '10 seconds'}
{'observation_date': '2019-05-16 12:00:00 Local', 'location_raw': 'Brownsville, TX, USA', 'shape': 'Oval'}
{'observation_date': '2001-07-29 23:59:00 Local', 'location_raw': 'Tucson, AZ, USA', 'shape': 'Unknown', 'duration_raw': '?'}


**Note:** The parser is tested on several reports to confirm that the labeled fields are extracted correctly across multiple records. 

### Apply Parser

In [39]:
# Apply parser to dataset --> converts extracted values into a structured dataframe

parsed_reports = nuforc["report"].apply(parse_nuforc_report)

parsed_df = pd.json_normalize(parsed_reports)

parsed_df.head()

,observation_date,location_raw,shape,duration_raw
0,2014-09-21 13:00:00 Local,"Huntsville, TX, USA",Rectangle,several seconds
1,2015-12-18 13:00:00 Local,"Sonoma, CA, USA",Sphere,2 minutes
2,2014-02-05 21:00:00 Local,"Hershey, PA, USA",Light,10 seconds
3,2019-05-16 12:00:00 Local,"Brownsville, TX, USA",Oval,NaN
4,2001-07-29 23:59:00 Local,"Tucson, AZ, USA",Unknown,?


**Notes**: The parser is applied to the entire dataset, producing a structured dataframe containing extracted observation date, location, shape, and duration fields for each report.

### Structured Dataset

The parsed dataframe contains the structured fields extracted from the NUFORC narrative 'report' text. At this stage, the data remains in an intermediate format and has not yet been cleaned, standardized, or merged back into the original dataset. 

In [40]:
# Inspect the structured dataframe created from the parsed report text
display(parsed_df.head(7))
display(parsed_df.tail(7))

parsed_df.shape

,observation_date,location_raw,shape,duration_raw
0,2014-09-21 13:00:00 Local,"Huntsville, TX, USA",Rectangle,several seconds
1,2015-12-18 13:00:00 Local,"Sonoma, CA, USA",Sphere,2 minutes
2,2014-02-05 21:00:00 Local,"Hershey, PA, USA",Light,10 seconds
3,2019-05-16 12:00:00 Local,"Brownsville, TX, USA",Oval,NaN
4,2001-07-29 23:59:00 Local,"Tucson, AZ, USA",Unknown,?
5,2015-07-04 21:30:00 Local,"Scotch Plains, NJ, USA",Circle,1 hour
6,2002-01-07 17:45:00 Local,"Richmond, VA, USA",Light,approx 3 sec.


,observation_date,location_raw,shape,duration_raw
147883,2006-07-30 16:40:00 Local,"Sunderland (UK/England), , United Kingdom",Circle,10 - 15 Seconds
147884,2019-11-19 05:11:00 Local,"Sandy Springs, GA, USA",Light,2mns
147885,2020-07-12 23:00:00 Local,"Roseburg, OR, USA",Circle,30 seconds
147886,2010-08-04 21:57:00 Local,"Greensburg, IN, USA",Triangle,30 seconds
147887,2018-08-08 16:00:00 Local,"Boonville, MO, USA",Unknown,sitting in the car
147888,1997-10-01 08:00:00 Local,"Oregon (rural), OR, USA",Rectangle,20 seconds
147889,2011-01-13 20:05:00 Local,"Basye, VA, USA",Diamond,2 min


(147890, 4)

**Interpretation:**

The structured dataframe confirms that the NUFORC report narratives were successfully parsed into structured fields.  
The dataset now contains extracted observation date, location, object shape, and duration attributes for each report.

Inspection of the first and last records confirms that the parser works consistently across the dataset.  
The dataframe contains **147,890 parsed reports and 4 extracted fields**, which will be cleaned and standardized in the next stage of the analysis.

## 9. Data Cleaning

Before performing transformations, the extracted fields are inspected to identify missing values, inconsistent formats, and other potential data quality issues. This step helps determine which cleaning and normalization procedures will be required.

### Dataframe Structure and Data Types

In [ ]:
# Inspect dataframe structure and data types
parsed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147890 entries, 0 to 147889
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   observation_date  147890 non-null  object
 1   location_raw      147889 non-null  object
 2   shape             141568 non-null  object
 3   duration_raw      140787 non-null  object
dtypes: object(4)
memory usage: 4.5+ MB


**Interpretation**: 
- Initial inspection shows that all extracted fields are currently stored as text rather than as cleaned analytical data types. 
- The `observation_date` field will need to be converted to a datetime format, while `duration_raw` will require normalization before it can be analyzed numerically.
- The output also shows that missing values are already present in some parsed fields, especially `shape` and `duration_raw`. 
- This confirms that additional cleaning steps will be necessary before the NUFORC data can be standardized and integrated into the final project dataset.

### Check Missing Values

In [43]:
# Count missing values in each column
parsed_df.isna().sum()

observation_date       0
location_raw           1
shape               6322
duration_raw        7103
dtype: int64

**Interpretation:**

- The missing value inspection shows that most parsed fields are well populated. 
- The `observation_date` field contains no missing values, while `location_raw` contains only a single missing entry.
- However, the `shape` and `duration_raw` fields contain a larger number of missing values. This is expected because some NUFORC reports do not include these labeled attributes in their narratives. These fields will require additional handling during the data cleaning stage.

### Inspect Unique Values

In [44]:
# Inspecting shape
parsed_df["shape"].value_counts().head(20)

shape
Light        27496
Circle       14367
Triangle     13087
Other        10063
Unknown      10022
Fireball      9881
Disk          8716
Sphere        7652
Oval          6369
Orb           5924
Formation     4865
Changing      3987
Cigar         3753
Rectangle     2610
Cylinder      2482
Flash         2440
Diamond       2116
Chevron       1742
Egg           1289
Teardrop      1238
Name: count, dtype: int64

In [47]:
# Inspecting duration
parsed_df["duration_raw"].value_counts().head(20)

duration_raw
5 minutes      8813
2 minutes      6427
10 minutes     6188
1 minute       5531
3 minutes      4737
30 seconds     3966
15 minutes     3707
10 seconds     3382
5 seconds      2953
20 minutes     2748
30 minutes     2512
1 hour         2225
15 seconds     2042
20 seconds     1945
4 minutes      1647
3 seconds      1577
2 seconds      1226
2 hours        1116
2-3 minutes    1065
1-2 minutes     918
Name: count, dtype: int64

**Interpreation**:

### Inspect Location Patterns

In [48]:
parsed_df["location_raw"].head(10)

0       Huntsville, TX, USA
1           Sonoma, CA, USA
2          Hershey, PA, USA
3      Brownsville, TX, USA
4           Tucson, AZ, USA
5    Scotch Plains, NJ, USA
6         Richmond, VA, USA
7       Youngstown, OH, USA
8         Bellevue, WA, USA
9             Dent, MN, USA
Name: location_raw, dtype: object

**Interpreation**:

## 10. Location Standardization

## 11. Geocoding

## 12. Exploratory Analysis

**Notes**: 